# 1) Import Libraries

In [15]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 2). Load dataset

In [16]:
df=pd.read_csv("/workspaces/Modelling_Credit_Defaults-Logistic_Regression-/data/lending_club_loan_data/loan.csv")
df.head()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 39717 entries, 0 to 39716
Columns: 111 entries, id to total_il_high_credit_limit
dtypes: float64(74), int64(13), str(24)
memory usage: 33.6 MB


/tmp/ipykernel_66710/788162185.py:1: DtypeWarning: Columns (0: next_pymnt_d) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv("/workspaces/Modelling_Credit_Defaults-Logistic_Regression-/data/lending_club_loan_data/loan.csv")


# 3). Data Inspection and Cleaning

In [17]:
df.info()
df.duplicated().sum() # Result: No duplicates
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 39717 entries, 0 to 39716
Columns: 111 entries, id to total_il_high_credit_limit
dtypes: float64(74), int64(13), str(24)
memory usage: 33.6 MB


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit
0,1077501,1296599,5000,5000,4975.0,36 months,10.65%,162.87,B,B2,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN
1,1077430,1314167,2500,2500,2500.0,60 months,15.27%,59.83,C,C4,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN
2,1077175,1313524,2400,2400,2400.0,36 months,15.96%,84.33,C,C5,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN
3,1076863,1277178,10000,10000,10000.0,36 months,13.49%,339.31,C,C1,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN
4,1075358,1311748,3000,3000,3000.0,60 months,12.69%,67.79,B,B5,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN


### Data Dictionary

In [18]:
lc_dkt=pd.read_excel("/workspaces/Modelling_Credit_Defaults-Logistic_Regression-/data/lending_club_loan_data/LCDataDictionary.xlsx")
lc_dkt

,LoanStatNew,Description
0,acc_now_delinq,The number of accounts on which the borrower i...
1,acc_open_past_24mths,Number of trades opened in past 24 months.
2,addr_state,The state provided by the borrower in the loan...
3,all_util,Balance to credit limit on all trades
4,annual_inc,The self-reported annual income provided by th...
...,...,...
148,settlement_amount,The loan amount that the borrower has agreed t...
149,settlement_percentage,The settlement amount as a percentage of the p...
150,settlement_term,The number of months that the borrower will be...
151,NaN,NaN


In [19]:
df.isna().sum()/len(df)*100

id                              0.000000
member_id                       0.000000
loan_amnt                       0.000000
funded_amnt                     0.000000
funded_amnt_inv                 0.000000
                                 ...    
tax_liens                       0.098195
tot_hi_cred_lim               100.000000
total_bal_ex_mort             100.000000
total_bc_limit                100.000000
total_il_high_credit_limit    100.000000
Length: 111, dtype: float64

In [20]:
def wrangle(df):
    # Pick columns that are not 100% null
    cols=["id", "member_id","loan_amnt","funded_amnt","funded_amnt_inv","term",
        "int_rate","installment","grade","sub_grade",
        "emp_title","emp_length","home_ownership","annual_inc",
        "verification_status","issue_d","loan_status","pymnt_plan",
        "url","desc","purpose","title",
        "zip_code","addr_state","dti","delinq_2yrs",
        "earliest_cr_line","inq_last_6mths","mths_since_last_delinq",
        "mths_since_last_record","open_acc","pub_rec","revol_bal",
        "revol_util","total_acc","initial_list_status","out_prncp",
        "out_prncp_inv","total_pymnt","total_pymnt_inv","total_rec_prncp",
        "total_rec_int","total_rec_late_fee","recoveries",
        "collection_recovery_fee","last_pymnt_d","last_pymnt_amnt",
        "next_pymnt_d","last_credit_pull_d","collections_12_mths_ex_med",
        "policy_code","application_type","acc_now_delinq",
        "chargeoff_within_12_mths","delinq_amnt","pub_rec_bankruptcies",
        "tax_liens"
        ]
    df=df[cols]
    
    drop_cols=[]
    
    # 1. Drop high cardinality features
    high_low_cad=["id","member_id","url","zip_code","pymnt_plan",
                  "chargeoff_within_12_mths","delinq_amnt","tax_liens",
                  "policy_code","initial_list_status","application_type",
                  "collections_12_mths_ex_med","acc_now_delinq", "title", "desc"]
    drop_cols.extend(high_low_cad)

    # 2. Drop columns with more than 50% Nulls
    null_cols=["mths_since_last_delinq", "mths_since_last_record", 
               "next_pymnt_d"]
    drop_cols.extend(null_cols)

    # 3. Drop Leaky features
    leaky_cols=["total_pymnt","total_pymnt_inv", "last_pymnt_amnt",
                "recoveries", "last_pymnt_d", "collection_recovery_fee",
                "total_rec_late_fee", "total_rec_int", "total_rec_prncp",
                "funded_amnt_inv", "out_prncp_inv", "out_prncp","funded_amnt",
                 "last_credit_pull_d", "issue_d", "sub_grade", "grade" ]


    drop_cols.extend(leaky_cols)

    # 4. Clean "int_rate" column
    df["int_rate"]=df["int_rate"].str.replace("%", "").astype(float)

    # 5. Clean "loan_status" column. Extract only loans whose final status is known and recast
    df=df[df["loan_status"] != "Current"]
    df["loan_status"]=df["loan_status"].map({"Fully Paid":0, "Charged Off":1 }).astype(int)

    # 6. Recast "states" to full names
    state_map = {"CA": "California","NY": "New York","FL": "Florida",
                "TX": "Texas","NJ": "New Jersey","IL": "Illinois",
                "PA": "Pennsylvania","VA": "Virginia","GA": "Georgia",
                "MA": "Massachusetts","OH": "Ohio","MD": "Maryland",
                "AZ": "Arizona","WA": "Washington","CO": "Colorado",
                "NC": "North Carolina","CT": "Connecticut","MI": "Michigan",
                "MO": "Missouri","MN": "Minnesota","NV": "Nevada",
                "SC": "South Carolina","WI": "Wisconsin","AL": "Alabama",
                "OR": "Oregon","LA": "Louisiana","KY": "Kentucky",
                "OK": "Oklahoma","KS": "Kansas","UT": "Utah",
                "AR": "Arkansas","DC": "District of Columbia","RI": "Rhode Island",
                "NM": "New Mexico","WV": "West Virginia","HI": "Hawaii",
                "NH": "New Hampshire","DE": "Delaware","MT": "Montana",
                "WY": "Wyoming","AK": "Alaska","SD": "South Dakota",
                "VT": "Vermont","MS": "Mississippi","TN": "Tennessee",
                "IN": "Indiana","ID": "Idaho","IA": "Iowa",
                "NE": "Nebraska","ME": "Maine"
                }
    df["addr_state"] = df["addr_state"].map(state_map)

    # 7. Convert "earliest_cr_line" column to datetime
    df["earliest_cr_line"] = pd.to_datetime(df["earliest_cr_line"], format="%b-%y")
    
    # 8. Clean "revol_util" column
    df["revol_util"]=df["revol_util"].str.replace("%", "").astype(float)


    df.drop(columns=drop_cols, inplace=True)

    # 9 Reset index
    #df = df.reset_index(drop=True)
    return df

In [21]:
df=wrangle(df)
df.head()

,loan_amnt,term,int_rate,installment,emp_title,emp_length,home_ownership,annual_inc,verification_status,loan_status,...,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,open_acc,pub_rec,revol_bal,revol_util,total_acc,pub_rec_bankruptcies
0,5000,36 months,10.65,162.87,NaN,10+ years,RENT,24000.0,Verified,0,...,27.65,0,1985-01-01,1,3,0,13648,83.7,9,0.0
1,2500,60 months,15.27,59.83,Ryder,< 1 year,RENT,30000.0,Source Verified,1,...,1.00,0,1999-04-01,5,3,0,1687,9.4,4,0.0
2,2400,36 months,15.96,84.33,NaN,10+ years,RENT,12252.0,Not Verified,0,...,8.72,0,2001-11-01,2,2,0,2956,98.5,10,0.0
3,10000,36 months,13.49,339.31,AIR RESOURCES BOARD,10+ years,RENT,49200.0,Source Verified,0,...,20.00,0,1996-02-01,1,10,0,5598,21.0,37,0.0
5,5000,36 months,7.90,156.46,Veolia Transportaton,3 years,RENT,36000.0,Source Verified,0,...,11.20,0,2004-11-01,3,9,0,7963,28.3,12,0.0


In [22]:
df.info()

<class 'pandas.DataFrame'>
Index: 38577 entries, 0 to 39716
Data columns (total 22 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   loan_amnt             38577 non-null  int64         
 1   term                  38577 non-null  str           
 2   int_rate              38577 non-null  float64       
 3   installment           38577 non-null  float64       
 4   emp_title             36191 non-null  str           
 5   emp_length            37544 non-null  str           
 6   home_ownership        38577 non-null  str           
 7   annual_inc            38577 non-null  float64       
 8   verification_status   38577 non-null  str           
 9   loan_status           38577 non-null  int64         
 10  purpose               38577 non-null  str           
 11  addr_state            38577 non-null  str           
 12  dti                   38577 non-null  float64       
 13  delinq_2yrs           38577 non-

In [23]:
df.pub_rec_bankruptcies.value_counts()

pub_rec_bankruptcies
0.0    36238
1.0     1637
2.0        5
Name: count, dtype: int64

In [24]:
df.nunique().sort_values(ascending=False)

emp_title               28027
revol_bal               21275
installment             15022
annual_inc               5215
dti                      2853
revol_util               1088
loan_amnt                 870
earliest_cr_line          524
int_rate                  370
total_acc                  82
addr_state                 50
open_acc                   40
purpose                    14
emp_length                 11
delinq_2yrs                11
inq_last_6mths              9
pub_rec                     5
home_ownership              5
pub_rec_bankruptcies        3
verification_status         3
term                        2
loan_status                 2
dtype: int64

In [25]:
(df.isna().sum()/len(df)*100).sort_values()

loan_amnt               0.000000
term                    0.000000
int_rate                0.000000
installment             0.000000
annual_inc              0.000000
home_ownership          0.000000
loan_status             0.000000
verification_status     0.000000
dti                     0.000000
delinq_2yrs             0.000000
purpose                 0.000000
addr_state              0.000000
earliest_cr_line        0.000000
inq_last_6mths          0.000000
pub_rec                 0.000000
open_acc                0.000000
revol_bal               0.000000
total_acc               0.000000
revol_util              0.129611
pub_rec_bankruptcies    1.806776
emp_length              2.677761
emp_title               6.185033
dtype: float64

In [26]:
#df.isna().sum()
col_dict={}
for col in df.columns:
    col_dict[col]=df[col].isna().sum()

for key, value in col_dict.items():
    if value < 39717:
        print(key)
    

loan_amnt
term
int_rate
installment
emp_title
emp_length
home_ownership
annual_inc
verification_status
loan_status
purpose
addr_state
dti
delinq_2yrs
earliest_cr_line
inq_last_6mths
open_acc
pub_rec
revol_bal
revol_util
total_acc
pub_rec_bankruptcies


In [27]:
df.info()

<class 'pandas.DataFrame'>
Index: 38577 entries, 0 to 39716
Data columns (total 22 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   loan_amnt             38577 non-null  int64         
 1   term                  38577 non-null  str           
 2   int_rate              38577 non-null  float64       
 3   installment           38577 non-null  float64       
 4   emp_title             36191 non-null  str           
 5   emp_length            37544 non-null  str           
 6   home_ownership        38577 non-null  str           
 7   annual_inc            38577 non-null  float64       
 8   verification_status   38577 non-null  str           
 9   loan_status           38577 non-null  int64         
 10  purpose               38577 non-null  str           
 11  addr_state            38577 non-null  str           
 12  dti                   38577 non-null  float64       
 13  delinq_2yrs           38577 non-

In [29]:
# Save clean df
df.to_csv("/workspaces/Modelling_Credit_Defaults-Logistic_Regression-/data/lending_club_loan_data/clean_data.csv", index=False)